# Forward model — $p(\log N_{HI} \mid \text{halo properties})$

Let's try to learn the **distribution** of the integrated HI column density of a sightline, given the host
halo and the geometry of the line of sight.

Not a point prediction. At fixed $(M_{200},\,b,\,v_{max})$ the column density is still random:
the orientation of the HI disc, the internal gas structure and the satellites all scatter it,
and none of that is visible in the features. So the model outputs a distribution, and it is
judged on whether that distribution is *calibrated*, not on $R^2$.

### The two parts

`log_n_hi = 0.0` in the pickles is not a measurement — it is a flag meaning "the integrated
column fell below $10^{17.2}$ and was zeroed". So the model is split in two:

$$p(\log N \mid \theta) \;=\;
\underbrace{f_{\rm cov}(\theta)}_{\textbf{Part 1} \;\rightarrow\; \text{classifier}}
\;\times\;
\underbrace{p(\log N \mid \theta,\ \texttt{is\_hcd})}_{\textbf{Part 2} \;\rightarrow\; \text{quantile regression}}$$

Part 1 is the covering fraction. Part 2 is the column-density distribution of the sightlines
that do make the cut. Each has a physical meaning on its own.

---
## Extra packages

```bash
conda activate fakespectra
pip install scikit-learn lightgbm ipympl
```

| Package | Used for |
|---|---|
| `scikit-learn` | `GroupShuffleSplit`, confusion matrix, pinball loss |
| `lightgbm` | the classifier and the quantile regressors |
| `ipympl` | the `%matplotlib widget` backend |

---
## 1. Setup

In [ ]:
import numpy as np

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path("..").resolve()                       # this notebook lives in training/
DATA_DIR  = REPO_ROOT / "data_TNG50-2_snap025_5runs"

# The five pickles differ only by the random seed used to draw the stratified halo
# sample. One of them is already 637 haloes and 231,231 sightlines, which is plenty
# for a first model. Raise N_RUNS only if you find yourself short of massive haloes:
# the top mass bin is sampled exhaustively, so extra runs repeat the same haloes there
# and only add new ones at low mass.
N_RUNS = 1

HUBBLE_H = 0.6774          # TNG cosmology; h is not stored in df.attrs
QUANTILES = [0.05, 0.16, 0.50, 0.84, 0.95]
TEST_FRACTION = 0.25       # fraction of HALOES held out, not of rows
SEED = 42

sys.path.append(str(REPO_ROOT / "tools"))
import radii_skewers as rs                             # add_concentration, categorize

In [ ]:
def matplotlib_default_config():
    font = {'family' : 'serif', 'weight' : 'normal', 'size' : 14}
    rcnew = {
        "mathtext.fontset" : "cm", "text.usetex": False, 'figure.frameon': True,
        'axes.linewidth': 2., "axes.titlesize" : 20, "axes.labelsize" : 20,
        "legend.fontsize" : 12, 'legend.fancybox': True, 'lines.linewidth': 2.5,
        'xtick.direction': 'in', 'ytick.direction': 'in',
        'xtick.top': True, 'ytick.right': True,
        'xtick.labelsize': 18, 'ytick.labelsize': 18
    }
    return font, rcnew

%matplotlib widget
font, rcnew = matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
plt.close('all')

---
## 2. Loading

Each pickle is ~1.1 GB, and nearly all of that is the three array columns (`flux`, `n_hi_col`,
`temp`: 396 float32 per row). The forward model touches none of them, so they are dropped
immediately after the read and only the 13 scalar columns are kept.

`df.attrs` carries the snapshot constants and does not survive the column subset, so it is
copied out into `meta` before anything else.

In [ ]:
SCALARS = ["halo_id", "halo_mass", "halo_r200", "halo_vmax", "redshift",
           "skewer_id", "axis", "position_tag", "r_multiplier", "impact_kpc",
           "log_n_hi", "is_hcd", "vel_peak_shift"]

files = sorted(DATA_DIR.glob("*.pkl"))[:N_RUNS]
frames, meta = [], None

for i, f in enumerate(files):
    print(f"[{i}] {f.name}", flush=True)
    d = pd.read_pickle(f)
    if meta is None:
        meta = dict(d.attrs)               # v_grid, redshift, length_unit_kpc, ...
    slim = d[SCALARS].copy()               # the three 396-pixel arrays are dropped here
    # column subsetting propagates df.attrs, which holds numpy arrays (v_grid, ...);
    # pd.concat compares attrs across frames and comparing arrays raises. The one
    # shared copy already lives in `meta`, so the per-frame attrs are dropped.
    slim.attrs = {}
    slim["run_id"] = i
    frames.append(slim)
    del d

df = pd.concat(frames, ignore_index=True)

print(f"\n{len(df):,} sightlines | {df.halo_id.nunique():,} haloes | "
      f"{df.is_hcd.mean():.1%} above the {meta['hcd_log_nhi_cut']} cut")
print(f"z = {meta['redshift']:.4f} | length_unit_kpc = {meta['length_unit_kpc']}")

In [ ]:
# The censoring is not uniform: it is driven almost entirely by impact parameter.
# This curve is what Part 1 has to reproduce, so it is worth looking at first.
cov = df.groupby("r_multiplier")["is_hcd"].mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogx(cov.index.values + 0.005, cov.values, "o-", color="crimson", ms=5)
ax.set_xlabel(r"$b\,/\,R_{200}$")
ax.set_ylabel(r"$f_{\rm cov}$")
ax.set_title("HCD covering fraction", fontsize=16)
ax.grid(alpha=0.3, ls=":")
fig.tight_layout()
plt.show()

---
## 3. Features

Six of them, and two choices worth spelling out.

**Only one mass variable.** `halo_r200` is fixed by `halo_mass` and $z$ through the definition
of $M_{200c}$, and $V_{200}=\sqrt{GM/R}$ follows from both. Giving the model all three would
split one piece of information across three columns and make the importances unreadable. The
*second* independent halo variable is the concentration $c_{200}$, which comes from
`halo_vmax` — the one structural quantity in the file that is not derivable from the others.

**Two versions of the impact parameter**, $b/R_{200}$ and $b$ in physical kpc. Which one the
model leans on is a physics question: it is the self-similarity test for the HI profile.

`axis` and `position_tag` go in as a null test. They encode the direction of the sightline,
which by symmetry of the box should carry no information — if they come out important,
something is wrong upstream.

Nothing produced by `rs.add_moments` (`eq_width`, `std_dev`, `T_hi`, ...) may be a feature:
those are functions of the spectrum, i.e. of the target.

In [ ]:
# c200 from the Klypin+01 Vmax/V200 inversion. NaN where Vmax/V200 <= 1, which is a
# real regime at z=3 and not an error -- LightGBM handles NaN natively, so it is kept.
rs.add_concentration(df, meta)

a_scale = 1.0 / (1.0 + float(meta["redshift"]))

df["log_M200"]   = np.log10(df["halo_mass"] * 1e10 / HUBBLE_H)            # log10 Msun
# +0.005 is half the smallest non-zero b, so the b=0 sightlines survive the log
df["log_b_r200"] = np.log10(df["r_multiplier"] + 0.005)
# impact_kpc is comoving kpc/h; the two factors turn it into physical kpc
df["log_b_kpc"]  = np.log10(df["impact_kpc"] * a_scale / HUBBLE_H + 0.01)
df["pos_code"]   = pd.Categorical(df["position_tag"]).codes

FEATURES = ["log_M200", "halo_c200", "log_b_r200", "log_b_kpc", "axis", "pos_code"]
TARGET   = "log_n_hi"

print(f"c200 defined for {np.isfinite(df.halo_c200).mean():.1%} of rows\n")
print(df[FEATURES].describe().T[["mean", "std", "min", "max"]].round(3).to_string())

---
## 4. Train / test split — by halo, never by row

231,231 rows, but only 637 haloes: 363 sightlines share each one. A random split over rows
would put the same halo on both sides and every score would be inflated by leakage.
`GroupShuffleSplit` on `halo_id` splits the haloes instead.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED)
i_tr, i_te = next(gss.split(df, groups=df["halo_id"]))
train, test = df.iloc[i_tr].copy(), df.iloc[i_te].copy()

assert not (set(train.halo_id) & set(test.halo_id)), "leakage: a halo is on both sides"

print(f"train  {len(train):>7,} rows  {train.halo_id.nunique():>4} haloes  "
      f"HCD {train.is_hcd.mean():.3f}")
print(f"test   {len(test):>7,} rows  {test.halo_id.nunique():>4} haloes  "
      f"HCD {test.is_hcd.mean():.3f}")

---
## 5. Part 1 — the covering fraction

A binary classifier for `is_hcd`. What matters is not accuracy but **calibration**: when the
model says 0.30 it has to be right 30% of the time, because that number *is* the covering
fraction. The Brier score measures exactly that (it is the mean squared error on the
probability), and the plot compares the predicted curve against the measured one.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, brier_score_loss

clf = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    min_child_samples=100,        # sightlines are correlated: keep leaves from getting tiny
    random_state=SEED, n_jobs=-1, verbose=-1,
)
clf.fit(train[FEATURES], train["is_hcd"].astype(int))
p_test = clf.predict_proba(test[FEATURES])[:, 1]

const = np.full(len(test), train.is_hcd.mean())     # "always predict the mean" reference
print(f"AUC          {roc_auc_score(test.is_hcd, p_test):.4f}")
print(f"Brier        {brier_score_loss(test.is_hcd, p_test):.4f}")
print(f"Brier (flat) {brier_score_loss(test.is_hcd, const):.4f}   <- what we have to beat")

In [ ]:
test["p_hcd"] = p_test

# Measured vs predicted covering fraction, in four mass bins. Points are the data,
# lines are the model averaged over the same sightlines.
edges = [10.0, 10.5, 11.0, 11.5, 13.5]
colors = plt.cm.viridis(np.linspace(0, 0.85, len(edges) - 1))

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for k in range(len(edges) - 1):
    sub = test[(test.log_M200 >= edges[k]) & (test.log_M200 < edges[k + 1])]
    if sub.empty:
        continue
    g = sub.groupby("r_multiplier").agg(obs=("is_hcd", "mean"), pred=("p_hcd", "mean"))
    x = g.index.values + 0.005
    ax.semilogx(x, g["obs"], "o", ms=5, color=colors[k],
                label=fr"${edges[k]} \leq \log M < {edges[k+1]}$")
    ax.semilogx(x, g["pred"], "-", lw=2, color=colors[k])

ax.set_xlabel(r"$b\,/\,R_{200}$")
ax.set_ylabel(r"$f_{\rm cov}$")
ax.set_title("points: measured   lines: model", fontsize=16)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, ls=":")
fig.tight_layout()
plt.show()

---
## 6. Part 2 — the column density of the absorbers that survive the cut

Trained **only** on `is_hcd == True`. For the rest, the pipeline zeroed the column, so the
true value does not exist and cannot be regressed.

One LightGBM per quantile with the pinball loss. Five quantiles give the median plus a 68%
and a 90% band, without assuming any shape for the distribution. The test is whether those
bands contain the nominal fraction of the true values.

In [ ]:
from sklearn.metrics import mean_pinball_loss

train_h, test_h = train[train.is_hcd], test[test.is_hcd]
y_test = test_h[TARGET].to_numpy()
print(f"HCD rows: train {len(train_h):,} | test {len(test_h):,}")

qmodels, Q = {}, []
for q in QUANTILES:
    m = lgb.LGBMRegressor(
        objective="quantile", alpha=q,
        n_estimators=600, learning_rate=0.05, num_leaves=63,
        min_child_samples=100, random_state=SEED, n_jobs=-1, verbose=-1,
    )
    m.fit(train_h[FEATURES], train_h[TARGET])
    qmodels[q] = m
    Q.append(m.predict(test_h[FEATURES]))

# The five fits are independent, so nothing forces q16 <= q50 <= q84. Sorting each
# row fixes the crossings; if the reported fraction is large the model is too flexible.
Q = np.vstack(Q)
print(f"rows with crossing quantiles: {np.mean(np.any(np.diff(Q, axis=0) < 0, axis=0)):.2%}")
Q = np.sort(Q, axis=0)
qp = {q: Q[i] for i, q in enumerate(QUANTILES)}

In [ ]:
for lo, hi in [(0.16, 0.84), (0.05, 0.95)]:
    emp = np.mean((y_test >= qp[lo]) & (y_test <= qp[hi]))
    print(f"nominal {hi-lo:.0%} band  ->  contains {emp:.1%} of the true values")

resid = y_test - qp[0.50]
print(f"\nmedian |residual|  {np.median(np.abs(resid)):.3f} dex")
print(f"RMS residual       {np.sqrt(np.mean(resid**2)):.3f} dex")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lo, hi = np.percentile(y_test, [0.5, 99.5])
axes[0].hexbin(y_test, qp[0.50], gridsize=55, bins="log", cmap="Blues",
               extent=(lo, hi, lo, hi))
axes[0].plot([lo, hi], [lo, hi], "k--", lw=2)
axes[0].axvline(20.3, color="crimson", ls=":", lw=2)      # DLA threshold
axes[0].axhline(20.3, color="crimson", ls=":", lw=2)
axes[0].set_xlabel(r"true $\log N_{HI}$")
axes[0].set_ylabel(r"predicted median")

axes[1].hist(resid, bins=80, color="royalblue", alpha=0.8, density=True)
axes[1].axvline(0, color="k", ls="--", lw=2)
axes[1].set_xlabel("residual [dex]")
axes[1].set_ylabel("density")
axes[1].grid(alpha=0.3, ls=":")
fig.tight_layout()
plt.show()

In [ ]:
# The model read as a profile: predict on a synthetic grid of b at fixed mass, so the
# bands can be laid over the measured quantiles.
b_grid = np.logspace(-2, np.log10(5.0), 60)
c_med = float(np.nanmedian(train_h["halo_c200"]))

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (m_lo, m_hi) in zip(axes, [(10.0, 10.7), (10.7, 11.4), (11.4, 13.5)]):
    sub = test_h[(test_h.log_M200 >= m_lo) & (test_h.log_M200 < m_hi)]
    if len(sub) < 50:
        continue
    logM = float(sub.log_M200.median())
    r200 = float((sub.halo_r200 * a_scale / HUBBLE_H).median())      # physical kpc
    grid = pd.DataFrame({"log_M200": logM, "halo_c200": c_med,
                         "log_b_r200": np.log10(b_grid + 0.005),
                         "log_b_kpc": np.log10(b_grid * r200 + 0.01),
                         "axis": 0, "pos_code": 1})[FEATURES]
    band = {q: qmodels[q].predict(grid) for q in QUANTILES}

    ax.fill_between(b_grid, band[0.05], band[0.95], color="royalblue", alpha=0.15)
    ax.fill_between(b_grid, band[0.16], band[0.84], color="royalblue", alpha=0.35)
    ax.plot(b_grid, band[0.50], color="navy", lw=2.5)

    bb = pd.cut(sub.r_multiplier, np.logspace(np.log10(0.008), np.log10(5.5), 15))
    g = sub.groupby(bb, observed=True)[TARGET].agg(
        med="median", q16=lambda s: s.quantile(0.16), q84=lambda s: s.quantile(0.84), n="size")
    g = g[g.n >= 20]
    ax.errorbar([iv.mid for iv in g.index], g["med"],
                yerr=[g["med"] - g["q16"], g["q84"] - g["med"]],
                fmt="o", color="crimson", ms=5, lw=1.5, capsize=3)

    ax.set_xscale("log")
    ax.set_xlabel(r"$b\,/\,R_{200}$")
    ax.set_title(fr"$\log M \in [{m_lo},\,{m_hi})$", fontsize=15)
    ax.axhline(20.3, color="grey", ls=":", lw=2)
    ax.grid(alpha=0.3, ls=":")

axes[0].set_ylabel(r"$\log N_{HI}$")
fig.tight_layout()
plt.show()

---
## 7. Does it beat binned medians?

The reference is not "predict the global median" — it is what the scatter plots already give
you: the median of $\log N_{HI}$ in cells of $(\log M_{200},\ \log b)$, measured on the
training haloes and applied to the test haloes.

If LightGBM does not clearly win, the conclusion is that $(M, b)$ is the whole story and the
extra features buy nothing. That would be a result, not a failure.

In [ ]:
m_edges = np.arange(9.9, 13.6, 0.2)
b_edges = np.arange(-2.4, 0.9, 0.15)

cells = lambda f: (pd.cut(f.log_M200, m_edges), pd.cut(f.log_b_r200, b_edges))

mtr, btr = cells(train_h)
cell_med = train_h.groupby([mtr, btr], observed=True)[TARGET].median()

mte, bte = cells(test_h)
base = np.array([cell_med.get(k, np.nan) for k in zip(mte, bte)])
base = np.where(np.isfinite(base), base, train_h[TARGET].median())   # empty cells

pin_base = mean_pinball_loss(y_test, base, alpha=0.5)
pin_lgbm = mean_pinball_loss(y_test, qp[0.50], alpha=0.5)

print(f"pinball q50   binned medians {pin_base:.4f}")
print(f"pinball q50   LightGBM       {pin_lgbm:.4f}")
print(f"improvement                  {100 * (1 - pin_lgbm / pin_base):+.1f}%")

---
## 8. The irreducible scatter floor

Every $(halo,\ b)$ pair was sampled with **12 sightlines**: 4 azimuthal offsets $\times$ 3
line-of-sight axes. All twelve share every feature the model can see, so their dispersion is
scatter that no halo-level feature can ever remove — it is orientation and internal structure.

**No model built on these features can produce a band narrower than this floor.** So the
comparison below is a direct statement about whether the model is done:

* ratio $\approx 1$ — the model is at the information limit of this feature set;
* ratio $> 1$ — there is information left in the features that the model is not using;
* ratio $< 1$ — the model is over-confident: its band is narrower than the physics allows.

In [ ]:
floor = (test_h.groupby(["halo_id", "r_multiplier"])[TARGET]
                .agg(sigma="std", n="size").reset_index())
floor = floor[floor.n >= 6]                       # need enough replicas for a std

width = test_h.assign(half=(qp[0.84] - qp[0.16]) / 2)
width = width.groupby(["halo_id", "r_multiplier"])["half"].mean().reset_index()
cmp = floor.merge(width, on=["halo_id", "r_multiplier"])

f_med, m_med = cmp.sigma.median(), cmp.half.median()
print(f"{len(cmp):,} (halo, b) groups with >= 6 HCD replicas")
print(f"irreducible floor   {f_med:.3f} dex")
print(f"model half-width    {m_med:.3f} dex")
print(f"ratio               {m_med / f_med:.2f}")

g = cmp.groupby("r_multiplier").agg(floor=("sigma", "median"), model=("half", "median"))
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.semilogx(g.index + 0.005, g["floor"], "o-", color="crimson", ms=5, label="irreducible floor")
ax.semilogx(g.index + 0.005, g["model"], "s-", color="royalblue", ms=5, label="model half-width")
ax.set_xlabel(r"$b\,/\,R_{200}$")
ax.set_ylabel("scatter [dex]")
ax.legend(fontsize=11)
ax.grid(alpha=0.3, ls=":")
fig.tight_layout()
plt.show()

---
## 9. Class check — with the whole distribution, not just the median

A first version of this section classified each sightline by its **predicted median** and
built a confusion matrix from that. The result looked damning for the tail classes — and the
test was rigged. With 0.5–0.9 dex of irreducible scatter, the median of a wide distribution
almost never lands in a narrow tail class, even when the model correctly assigns it a 20%
probability: `Large_DLA` gets starved and `Small_DLA` inflated *by construction*. A point
estimate cannot represent a distribution.

Two tests that use what the model actually predicts:

1. **Expected counts.** Every sightline gets a probability for each class — `Forest` from the
   classifier, the four HCD classes by reading the CDF (interpolated through the five fitted
   quantiles) at the class edges. Summing those probabilities over the test set gives the
   number of sightlines the model *expects* in each class, to set against the observed number.
   If the ratios are near 1, the model has the class demographics right even where the median
   never visits.
2. **Sampled confusion matrix.** Draw one class per sightline from its probabilities and build
   the matrix from the draws. Bleed into *neighbouring* classes is the irreducible scatter
   doing its job — the classes are just bins of a continuous variable. Bleed into distant
   classes is not.

In [ ]:
EDGES = np.array([17.2, 19.0, 20.3, 21.0])       # the class boundaries in log N_HI
qs = np.asarray(QUANTILES)

# Fitted quantiles for EVERY test sightline this time (Section 6 only needed the HCD ones),
# sorted per row so they define a monotonic CDF.
Qa = np.sort(np.column_stack([qmodels[q].predict(test[FEATURES]) for q in QUANTILES]), axis=1)
ph = test["p_hcd"].to_numpy()

# CDF at each class edge: linear interpolation through the five (value, quantile) pairs.
# Outside the fitted range the CDF is clamped to 0 or 1.
F = np.column_stack([[np.interp(e, Qa[i], qs, left=0.0, right=1.0) for i in range(len(Qa))]
                     for e in EDGES])

# Hurdle model assembled: P(Forest) from the classifier, the rest from the conditional CDF.
probs = np.column_stack([
    1.0 - ph,                     # Forest
    ph * (F[:, 1] - F[:, 0]),     # LLS        17.2 - 19.0
    ph * (F[:, 2] - F[:, 1]),     # Sub_DLA    19.0 - 20.3
    ph * (F[:, 3] - F[:, 2]),     # Small_DLA  20.3 - 21.0
    ph * (1.0 - F[:, 3]),         # Large_DLA  >= 21.0
])
probs = np.clip(probs, 0.0, None)
probs /= probs.sum(axis=1, keepdims=True)

labels = list(rs.CATEGORY_LABELS)
obs = rs.categorize(test[TARGET]).value_counts().reindex(labels).to_numpy()
exp = probs.sum(axis=0)

print(f"{'class':<11}{'expected':>10}{'observed':>10}{'ratio':>8}")
for lab, e, o in zip(labels, exp, obs):
    print(f"{lab:<11}{e:>10,.0f}{o:>10,}{e / o:>8.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix

# One draw per sightline from its class probabilities (inverse CDF over the 5 classes).
rng = np.random.default_rng(SEED)
draw = (rng.random(len(test))[:, None] < probs.cumsum(axis=1)).argmax(axis=1)
y_pred = np.array(labels)[draw]
y_true = rs.categorize(test[TARGET]).astype(str)

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=12)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=12)
ax.set_xlabel("predicted (sampled from the model)")
ax.set_ylabel("true")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}\n({cm[i, j]:,})", ha="center", va="center",
                fontsize=10, color="white" if cm_norm[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax, label="fraction of the true class")
fig.tight_layout()
plt.show()

print("recall per class (the diagonal):")
for lab, r, n in zip(labels, np.diag(cm_norm), cm.sum(axis=1)):
    print(f"  {lab:<10} {r:.3f}   ({n:,} sightlines)")
plt.show()

---
## 10. What the model is using

Gain importance, in percent. Read it against Section 3: `log_M200` and the two impact
parameters should dominate, `halo_c200` mattering would be the interesting result, and
`axis`/`pos_code` (grey) should be near zero — they are the null test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, model, name in [(axes[0], clf, "Part 1: covering fraction"),
                        (axes[1], qmodels[0.50], "Part 2: median")]:
    imp = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES)
    imp = (100 * imp / imp.sum()).sort_values()
    ax.barh(imp.index, imp.values,
            color=["lightgrey" if f in ("axis", "pos_code") else "royalblue" for f in imp.index])
    ax.set_xlabel("gain importance [%]")
    ax.set_title(name, fontsize=15)
    ax.grid(alpha=0.3, ls=":", axis="x")
    for y, v in enumerate(imp.values):
        ax.text(v + 1, y, f"{v:.1f}", va="center", fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# axis and pos_code picked up a few percent of gain in Part 2. Gain importance
# rewards whatever helps fit residual noise, so before reading anything physical into
# that: retrain the median without them and see whether the loss actually moves.
FEAT_PHYS = [f for f in FEATURES if f not in ("axis", "pos_code")]
m_phys = lgb.LGBMRegressor(
    objective="quantile", alpha=0.5,
    n_estimators=600, learning_rate=0.05, num_leaves=63,
    min_child_samples=100, random_state=SEED, n_jobs=-1, verbose=-1,
).fit(train_h[FEAT_PHYS], train_h[TARGET])

pin_phys = mean_pinball_loss(y_test, m_phys.predict(test_h[FEAT_PHYS]), alpha=0.5)
print(f"pinball q50 with axis/pos_code     {pin_lgbm:.4f}")
print(f"pinball q50 without them           {pin_phys:.4f}")
print("-> a difference in the 3rd decimal means the importance was noise-fitting;")
print("   a real degradation means the box has a preferred direction to investigate")

---
## 11. Save

`FEATURES` is saved next to the models on purpose: applying them with a different column
order silently returns nonsense instead of raising.

In [ ]:
import pickle

Path("models").mkdir(exist_ok=True)
with open("models/forward_hurdle_v0.pkl", "wb") as fh:
    pickle.dump({"clf": clf, "qmodels": qmodels, "FEATURES": FEATURES,
                 "QUANTILES": QUANTILES, "hubble_h": HUBBLE_H,
                 "redshift": float(meta["redshift"]), "n_runs": N_RUNS}, fh)
print("saved -> models/forward_hurdle_v0.pkl")

---
## Where to go next

Roughly in order of value:

1. **Orientation.** Add $\cos\alpha$ between the sightline and the halo's gas angular
   momentum, read from the FoF catalogue. Section 8 tells you the size of the prize: the gap
   between the model half-width and the floor is the budget, and HI at $z=3$ is anisotropic
   enough that this is the most likely single feature to close it.
2. **Uncensor.** Re-extract one realization with `zero_below_cut=False`. Recovering the
   sub-17.2 columns turns the hurdle into a truncated regression and gives the full
   $f(N_{HI})$ instead of only its high end.
3. **Mixture Density Network.** Replaces Section 6 with a real conditional density: no
   crossing quantiles, a proper likelihood, a PIT histogram instead of two coverage numbers.
4. **Resolution transfer.** Train on TNG50-3, test on TNG50-2 — the same machinery with the
   split on simulation instead of halo.
5. **The observables.** $\sigma_{\rm DLA}(M)$ by integrating Part 1 over $b$, then $dN/dX$ and
   $f(N_{HI})$ against the halo mass function — reweighted, since the stratified sampling
   flattened it on purpose.